# 14. Statistics reported in the abstract

Numbers in the abstract that are not produced by notebooks 05–13:
- tag rates at 4–12 m for mutually exclusive possession histories, and the distance-adjusted odds ratio
  (match-level cluster bootstrap)
- tag rates for passes with no defender within 2.74 m, by the label recorded at reception
- majority-class baseline of the classifier sample

Inputs: outputs of notebooks 05, 10 and 11.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import SplineTransformer

OUT = os.path.join("..", "outputs")
KEY = ["game_id", "pe_id"]
E = pd.read_csv(os.path.join(OUT, "pressure_events.csv.gz"))
X = E[E.label.isin(["P", "N"])].merge(pd.read_csv(os.path.join(OUT, "possession_exposure.csv.gz")), on=KEY) \
      .merge(pd.read_csv(os.path.join(OUT, "reception_labels.csv.gz")), on=KEY, how="left")
X["y"] = (X.label == "P").astype(int)
# mutually exclusive histories between reception and pass
X["history"] = np.select([X.d_min < 2.74, X.init_pressure.eq("P")],
                         ["defender within 2.74 m", "pressure recorded at reception"], "no pressure history")
print(f"pressured vs not pressured passes with tracking: {len(X):,}")

pressured vs not pressured passes with tracking: 52,389


## 1. Passes with the nearest defender 4–12 m away at the pass

In [2]:
far = X[(X.d_end >= 4) & (X.d_end < 12)]
display(far.groupby("history").y.agg(rate="mean", n="size").round(3))
ex = far[far.history.isin(["defender within 2.74 m", "no pressure history"])].copy()
ex["exposed"] = (ex.history == "defender within 2.74 m").astype(int)


def odds_ratio(d):
    # logistic regression of the label on exposure, adjusted for distance at the pass (cubic spline)
    Z = np.column_stack([d.exposed, SplineTransformer(n_knots=4).fit_transform(d[["d_end"]])])
    return float(np.exp(LogisticRegression(C=np.inf, max_iter=5000).fit(Z, d.y).coef_[0][0]))


games = ex.game_id.unique()
idx = {g: np.flatnonzero(ex.game_id.values == g) for g in games}
rng = np.random.default_rng(0)
boot = [odds_ratio(ex.iloc[np.concatenate([idx[g] for g in rng.choice(games, len(games))])]) for _ in range(300)]
print(f"distance-adjusted odds ratio = {odds_ratio(ex):.2f} "
      f"(95% CI {np.percentile(boot, 2.5):.2f}–{np.percentile(boot, 97.5):.2f}, match-level cluster bootstrap)")

,rate,n
history,,
defender within 2.74 m,0.342,1037
no pressure history,0.091,20279
pressure recorded at reception,0.896,1700


distance-adjusted odds ratio = 4.42 (95% CI 3.68–5.29, match-level cluster bootstrap)


## 2. Passes with no defender within 2.74 m during the possession, by the label recorded at reception

In [3]:
never = X[(X.d_min >= 2.74) & X.init_pressure.isin(["P", "N"])]
display(never.groupby("init_pressure").y.agg(rate="mean", n="size").round(3))
carried = never[never.init_pressure == "P"]
print(f"share of all pressured labels: {carried.y.sum() / X.y.sum():.3f}")

,rate,n
init_pressure,,
N,0.127,29976
P,0.912,3466


share of all pressured labels: 0.143


## 3. Majority-class baseline of the classifier sample (notebook 13)

In [4]:
print(f"no-pressure share = {1 - X.y.mean():.3f} (n = {len(X):,})")

no-pressure share = 0.577 (n = 52,389)
